# IndexTTS-2.5 — Colab TTS Test

Direct IndexTTS-2.5 inference with Japanese voice cloning, emotion control, Kana reading control, latency, RTF, and VRAM measurements.

IndexTTS-2.5 requires a reference voice recording. Only upload/use a voice you have permission to clone.

Recommended: a fresh Colab GPU runtime. IndexTTS manages its supported Python environment through `uv`.


In [ ]:
# 1) GPU + Python
!nvidia-smi
import platform
print("Colab Python:", platform.python_version())


In [ ]:
# 2) Clone IndexTTS and create its official uv environment
!rm -rf /content/index-tts
!git clone -q https://github.com/index-tts/index-tts.git /content/index-tts
%cd /content/index-tts
!pip -q install -U uv
!uv sync


In [ ]:
# 3) Download IndexTTS-2.5 weights
%cd /content/index-tts
!uv run --with "huggingface_hub[cli,hf_xet]" python -c "from huggingface_hub import snapshot_download; snapshot_download('IndexTeam/IndexTTS-2.5', local_dir='checkpoints')"
!ls -lh checkpoints | head


In [ ]:
# 4) Upload a reference voice WAV/MP3
from google.colab import files
import os
uploaded = files.upload()
REF_AUDIO = os.path.join('/content/index-tts', next(iter(uploaded.keys())))
print("Reference:", REF_AUDIO)


In [ ]:
# 5) Write the inference + benchmark runner inside the IndexTTS uv environment
%%writefile /content/index-tts/colab_index_test.py
import os, time, json
import torch
import soundfile as sf
from indextts.infer_v2_5 import IndexTTS2

REF_AUDIO = os.environ['REF_AUDIO']
TEXT = os.environ.get('TEXT', '今日は一緒に日本語を練習しましょう。好きな食べ物について教えてください。')
OUT = os.environ.get('OUT', '/content/index_japanese.wav')

print('Loading IndexTTS-2.5...')
t_load = time.perf_counter()
tts = IndexTTS2(cfg_path='checkpoints/config.yaml', model_dir='checkpoints', use_bf16=True)
print(f'Load time: {time.perf_counter()-t_load:.2f}s')

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
torch.cuda.synchronize()
t0 = time.perf_counter()
tts.infer(
    spk_audio_prompt=REF_AUDIO,
    text=TEXT,
    lang='ja',
    output_path=OUT,
    emo_vector=[0.15, 0.0, 0.0, 0.0, 0.0, 0.0, 0.05, 0.40],
    emo_alpha=0.65,
    verbose=True,
)
torch.cuda.synchronize()
elapsed = time.perf_counter() - t0
audio, sr = sf.read(OUT)
duration = len(audio) / sr
peak = torch.cuda.max_memory_allocated() / 1024**3
stats = {'generation_s': elapsed, 'audio_s': duration, 'rtf': elapsed/duration, 'peak_vram_gb': peak, 'sample_rate': sr, 'output': OUT}
print('COLAB_STATS=' + json.dumps(stats))


In [ ]:
# 6) Generate Japanese speech
import os, subprocess
from IPython.display import Audio, display
env = os.environ.copy()
env['REF_AUDIO'] = REF_AUDIO
env['TEXT'] = '今日は一緒に日本語を練習しましょう。好きな食べ物について教えてください。'
env['OUT'] = '/content/index_japanese.wav'
p = subprocess.run(['uv', 'run', 'python', 'colab_index_test.py'], cwd='/content/index-tts', env=env, text=True, capture_output=True)
print(p.stdout)
if p.returncode != 0:
    print(p.stderr)
    raise RuntimeError('IndexTTS inference failed')
display(Audio('/content/index_japanese.wav'))


In [ ]:
# 7) Japanese Kana pronunciation-control example
# Same kanji, two forced readings.
env = os.environ.copy()
env['REF_AUDIO'] = REF_AUDIO
env['TEXT'] = '彼は料理が<上手|じょうず>ですが、囲碁では相手のほうが<上手|うわて>でした。'
env['OUT'] = '/content/index_kana_control.wav'
p = subprocess.run(['uv', 'run', 'python', 'colab_index_test.py'], cwd='/content/index-tts', env=env, text=True, capture_output=True)
print(p.stdout)
if p.returncode != 0:
    print(p.stderr)
    raise RuntimeError('IndexTTS inference failed')
display(Audio('/content/index_kana_control.wav'))


## AIKO test ideas

Test short vocabulary, complete conversational sentences, emotional lines, difficult kanji readings, and longer paragraphs. Record RTF, peak VRAM, pronunciation accuracy, emotion, and speaker consistency.

For the API latency/concurrency test, use `IndexTTS25_API_Colab.ipynb` in a fresh runtime.
